In [1]:
"""PEPS DMRG notebook experiments."""
from IPython.display import display
import quimb.tensor as qtn

import pepsy as py
core = py.core

import torch


In [2]:
py.reg_complex_svd_torch()
py.reg_complex_svd_jax()

In [3]:
to_backend = core.backend_torch(dtype=torch.complex128)
optimizer = core.build_optimizer(progbar=False, directory="cash/", parallel=False)
coptimizer = core.build_compressed_optimizer(progbar=False, directory=None)


In [4]:
Lx, Ly, cyclic = 4, 4, False
edges = qtn.edges_2d_square(Lx=Lx, Ly=Ly, cyclic=cyclic)
sites = sorted({(site,) for edge in edges for site in edge})
num_sites = len(sites)
print(num_sites)


16


In [5]:
j_coupling, field_h, dt = 1.0, 0.5, 0.25

rx = py.rx(to_backend(-field_h * dt))
rzz = py.rzz(to_backend(-j_coupling * 2 * dt))

In [6]:
gates = []

for site in sites:
     gates.append( (site, rx) )

for edge in edges:
     gates.append( (edge, rzz) )


# gates += [ (((0,0),(Lx-1,Ly-1)),rzz)  ] 

In [7]:
pepo = py.gate_to_pepo(gates, cyclic=cyclic)
pepo.show()

  ╱ 2  ╱ 2  ╱ 2  ╱
 ●━━━━●━━━━●━━━━●
╱┃2  ╱┃2  ╱┃2  ╱┃2  
 ┃╱ 2 ┃╱ 2 ┃╱ 2 ┃╱
 ●━━━━●━━━━●━━━━●
╱┃2  ╱┃2  ╱┃2  ╱┃2  
 ┃╱ 2 ┃╱ 2 ┃╱ 2 ┃╱
 ●━━━━●━━━━●━━━━●
╱┃2  ╱┃2  ╱┃2  ╱┃2  
 ┃╱ 2 ┃╱ 2 ┃╱ 2 ┃╱
 ●━━━━●━━━━●━━━━●
╱    ╱    ╱    ╱    


In [8]:
peps = qtn.PEPS.rand(Lx=Lx, Ly=Ly, bond_dim=2, seed=666)
peps = core.ps_to_peps(Lx, Ly, dtype="complex128", theta=0.)
peps.apply_to_arrays(to_backend)


In [9]:
(peps.H & peps).contract(all, optimize=optimizer)

tensor(1.+0.j, dtype=torch.complex128)

In [10]:
peps_norm_opt = {"opt":optimizer,  "copt":coptimizer, "sequence": ["ymin", "ymax"]}
peps_norm_opt |= {"max_separation":1, "cutoff":1.e-12, "progbar":True}
peps_norm_opt |= {"mode":"mps", "mode_":"full-bond", "chi":64}

In [11]:
gopt = py.GlobalOptimizer(
    state=peps,
    # or mode="exact" with opt="auto-hq"
)

In [12]:
peps = gopt.normalize(mode="exact", opt=optimizer, progbar=True)
gopt.norm()

tensor(1.+0.j, dtype=torch.complex128)

In [13]:
(peps.H & peps).contract(all, optimize=optimizer)

tensor(1.+0.j, dtype=torch.complex128)

In [14]:
depth = 40
for step_ in range(depth):
    peps = py.gate_2d(peps, gates)
    peps_target = peps.copy()
    peps.compress_all_(max_bond=3, cutoff=1.e-12)
    gopt = py.GlobalOptimizer(state=peps, state_target=peps_target)
    peps = gopt.normalize(state=peps, mode="exact", chi=32, opt=optimizer, progbar=False)
    loss_val = gopt.loss(state=peps, mode="exact", chi=32, opt=optimizer, progbar=False)
    print(step_, gopt.norm(state=peps),  float(loss_val))


0 tensor(1.0000+9.8597e-18j, dtype=torch.complex128) 2.4424906541753444e-15
1 tensor(1.0000-1.5598e-15j, dtype=torch.complex128) 9.323652960802065e-13
2 tensor(1.0000-1.3668e-15j, dtype=torch.complex128) 3.8388137113543053e-10
3 tensor(1.0000-2.5835e-15j, dtype=torch.complex128) 1.2609455923850987e-08
4 tensor(1.0000-2.9419e-15j, dtype=torch.complex128) 3.7624997084595435e-07
5 tensor(1.0000+3.7411e-15j, dtype=torch.complex128) 3.0919397498996304e-06
6 tensor(1.0000+8.0002e-16j, dtype=torch.complex128) 6.955996705082512e-06
7 tensor(1.0000+2.6400e-15j, dtype=torch.complex128) 1.146991300560174e-05
8 tensor(1.0000+9.7798e-15j, dtype=torch.complex128) 3.73602174253973e-05
9 tensor(1.0000-4.7184e-15j, dtype=torch.complex128) 4.039363018282227e-05
10 tensor(1.0000+1.8648e-15j, dtype=torch.complex128) 3.34439408680165e-05
11 tensor(1.0000-2.1129e-15j, dtype=torch.complex128) 1.515240185512301e-05
12 tensor(1.0000-4.4215e-15j, dtype=torch.complex128) 2.420556249727035e-05
13 tensor(1.0000+2.

In [15]:
loss_peps_opt = {"opt":optimizer,  "copt":coptimizer, "sequence": ["ymin", "ymax", "xmin", "xmax"]}
loss_peps_opt |= {"max_separation":1, "cutoff":1.e-12, "progbar":False}
loss_peps_opt |= {"mode":"exact", "mode_":"mps", "chi":64}

In [16]:
tn_out = gopt.optimize(
    n=2000,
    optimizer="lbfgs",
    loss_kwargs = loss_peps_opt,
)

+0.000005540297 [best: +0.000005540297] :   9%|▉         | 186/2000 [00:24<04:00,  7.54it/s]


In [17]:
(peps.H & peps).contract(all, optimize=optimizer), (peps_target.H & peps_target).contract(all, optimize=optimizer)

(tensor(1.0000-8.0776e-16j, dtype=torch.complex128),
 tensor(1.0000+4.1926e-16j, dtype=torch.complex128))

In [18]:
sweeper = py.SweepOptimizer(
    state=peps.copy(),
    state_target=peps_target.copy(),
    chi=60,
    contraction_opt=optimizer,
    renormalize_state=True,
    renormalize_kwargs={
        'n_iter': 5,
        'direction': 'y',
        'max_separation': 1,
        'progress': False,
        'track_boundary_fidelity': False,
    },
)


In [21]:
sweeper.set_optimize_kwargs(
        axes=('y', 'x'),
        n_round_trips=1,
        optimizer='nlopt-lbfgs',
        optimizer_options={
            'algorithm': 'LBFGS',
            'lr': 1e-2,
            'n_steps': 50,
            'maxeval': 100,
        },
        env_n_iter=10,
        progress=True,
        renormalize=True,
    )

In [22]:
sweep_result = sweeper.run(n_cycles=5)

bdy_dmrg::   0%|                              | 0/100 [00:00<?, ?it/s]/Users/rezah/Documents/pepsy/src/pepsy/optimize_sweep.py:1182: UserWarning: solver='nlopt-lbfgs' uses NLopt on CPU float64 parameter vectors. Tune NLopt controls (algorithm/maxeval/ftol_rel/xtol_rel) for your problem.
  axis_runs = self.optimize_axis(
bdy_dmrg::   0%|                              | 0/100 [00:47<?, ?it/s]2.39s/it, loss=0.000004, t_bdy=0.00s, t_opt=0.25s, slice=x_fwd_0]


KeyboardInterrupt: 

In [ ]:
sweeper.loss[0], sweeper.loss[-1]

In [ ]:
# # Increase boundary chi, then rerun sweeps (two equivalent options).
# # Option 1: explicit API
# sweeper.set_chi(80)

# # Option 2: pass chi directly in run()
# sweep_result_chi = sweeper.run(chi=80, n=2, debug=False, progbar=True)
# print("loss(before->after) @chi=80:", sweep_result_chi.loss_before, "->", sweep_result_chi.loss_after)
